# Экспорт модели с HF Hub в Kaggle Dataset

Сабмит-ноутбук работает **без интернета**, поэтому модели надо положить в Kaggle Dataset.
Этот ноутбук (интернет **On**) скачивает финальные модели с HF Hub в `/kaggle/working/models`.

**Вариант А — один датасет на обе модели:** оставь обе в `RUN_NAMES`, выполни,
затем **Output → Create Dataset** из `/kaggle/working/models`.

**Вариант Б — отдельный датасет на каждую модель** (как ты хочешь):
1. Поставь `RUN_NAMES = ["byt5-full-seed13"]`, выполни → Create Dataset (напр. `akkadian-seed13`).
2. Поменяй на `RUN_NAMES = ["byt5-full-seed42"]`, выполни ещё раз → Create Dataset (`akkadian-seed42`).
3. В сабмит-ноутбуке подключи оба датасета и впиши пути в `MODEL_DIRS`.

Настройка: Internet → On, Secret `HF_TOKEN` (Attach).

In [ ]:
# какие обученные модели выгрузить (run_name из конфигов).
# Для финального сабмита нужен ансамбль -> обе full-модели.
RUN_NAMES = ["byt5-full-seed13", "byt5-full-seed42"]
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login, whoami, snapshot_download
login(os.environ["HF_TOKEN"])
USER = whoami()["name"]
print("HF user:", USER)

In [ ]:
import shutil
from pathlib import Path

# файлы, нужные для инференса (веса + конфиг + токенайзер ByT5)
PATTERNS = ["config.json", "generation_config.json", "*.safetensors",
            "pytorch_model.bin", "tokenizer_config.json", "special_tokens_map.json",
            "added_tokens.json", "spiece.model"]

for run in RUN_NAMES:
    repo = f"{USER}/akkadian-{run}"
    dst = Path("/kaggle/working/models") / run
    dst.mkdir(parents=True, exist_ok=True)
    # финальная модель лежит в корне репозитория (push_to_hub), не в last-checkpoint/
    snapshot_download(repo_id=repo, allow_patterns=PATTERNS,
                      local_dir=str(dst), ignore_patterns=["last-checkpoint/*"])
    print(run, "->", sorted(p.name for p in dst.iterdir()))

In [ ]:
# проверка: модель грузится и переводит (на стоковом transformers Kaggle, без нашего пакета)
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
run = RUN_NAMES[0]
d = f"/kaggle/working/models/{run}"
tok = AutoTokenizer.from_pretrained(d)
m = AutoModelForSeq2SeqLM.from_pretrained(d).eval()
enc = tok("um-ma kà-ru-um kà-ni-iš-ma", return_tensors="pt")
print(tok.decode(m.generate(**enc, num_beams=4, max_new_tokens=64)[0], skip_special_tokens=True))